In [28]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import graphviz
import torch

In [29]:
STORAGE_PATH = './figures/'
os.makedirs(STORAGE_PATH, exist_ok = True)

In [30]:
def create_grey_to_black_colormap():
  color_dictionary = {
    'red': [
      (0.0, 0.75, 0.75),
      (1.0, 0.0, 0.0)
    ],
    'green': [
      (0.0, 0.75, 0.75),
      (1.0, 0.0, 0.0)
    ],
    'blue': [
      (0.0, 0.75, 0.75),
      (1.0, 0.0, 0.0)
    ]
  }
    
  return mcolors.LinearSegmentedColormap('GreyToBlack', color_dictionary)

def create_yellow_colormap():
  light_yellow = mcolors.hex2color('#FFF9E2')
  dark_yellow = mcolors.hex2color('#FFD500')
    
  color_dictionary = {
    'red':   [(0.0, light_yellow[0], light_yellow[0]), (1.0, dark_yellow[0], dark_yellow[0])],
    'green': [(0.0, light_yellow[1], light_yellow[1]), (1.0, dark_yellow[1], dark_yellow[1])],
    'blue':  [(0.0, light_yellow[2], light_yellow[2]), (1.0, dark_yellow[2], dark_yellow[2])]
  }

  return mcolors.LinearSegmentedColormap('YellowGradient', color_dictionary)

def create_color_gradient(values, colormap):
  norm = plt.Normalize(np.min(values), np.max(values))
  normalized_values = norm(values)    
  rgb_colors = colormap(normalized_values)
  return [mcolors.to_hex(color) for color in rgb_colors]

In [31]:
grey_to_black_cmap = create_grey_to_black_colormap()
light_to_dark_yellow_cmap = create_yellow_colormap()

In [32]:
def visualize_graph(path, method, filename, edge):
  graph = torch.load(os.path.join(path, method, filename))
  graph_df = pd.DataFrame({
    'from_index' : graph['edge_index'][0],
    'from_token' : graph['from_tokens'],
    'to_index' : graph['edge_index'][1],
    'to_token' : graph['to_tokens'],
  })
  for dimension in range(graph['edge_attr'].size(1)):
    graph_df[f'edge_weight_{dimension}'] = graph['edge_attr'][:, dimension].tolist()
  
  weight_columns = [f'edge_weight_{dimension}'for dimension in range(graph['edge_attr'].size(1))]
  incoming = graph_df[['to_index', 'to_token'] + weight_columns].rename(columns = {'to_index' : 'id', 'to_token' : 'token'}).groupby(['id', 'token'])[weight_columns].sum()
  outgoing = graph_df[['from_index', 'from_token'] + weight_columns].rename(columns = {'from_index' : 'id', 'from_token' : 'token'}).groupby(['id', 'token'])[weight_columns].sum()
  nodes = incoming.add(outgoing, fill_value = 0).reset_index()
  
  edge_colors = create_color_gradient(graph_df[f'edge_weight_{edge}'], grey_to_black_cmap)
  node_colors = create_color_gradient(nodes[f'edge_weight_{edge}'], light_to_dark_yellow_cmap)

  graph = graphviz.Digraph(f'{filename.replace(".pt", "")}-{edge}', engine = 'dot') # engine = 'circo', 'twopi', 'dot
  graph.graph_attr['dpi'] = '300'

  min_size = 0.5
  max_size = 2.0
  min_weight = nodes[f'edge_weight_{edge}'].min()
  max_weight = nodes[f'edge_weight_{edge}'].max()
  for i, node in nodes.iterrows():
    normalized = (node[f'edge_weight_{edge}'] - min_weight) / (max_weight - min_weight) if max_weight != min_weight else 0.5
    size = min_size + normalized * (max_size - min_size)
    graph.node(str(node['id']), label = node['token'], color = '#363636', style = 'filled', fillcolor = node_colors[i], shape = 'oval', width = str(2.5 * size), height = str(size), fixedsize = 'true')

  for i, row in graph_df.iterrows():
    graph.edge(str(row['from_index']), str(row['to_index']), arrowsize = '0.5', color = edge_colors[i])
  os.makedirs(os.path.join(STORAGE_PATH, method), exist_ok = True)
  graph.render(directory = os.path.join(STORAGE_PATH, method, f'{filename.replace(".pt", "")}-{edge}'), format = 'png', view = True)

In [33]:
visualize_graph(path = './graphs/', method = 'sliding_windows', filename = 'SST-2-test-20.pt', edge = 0)

In [34]:
visualize_graph(path = './graphs/', method = 'attention_distillation', filename = 'SST-2-test-20.pt', edge = 0)

In [ ]:
def visualize_graph(path, method, filename, edge):
  graph = torch.load(os.path.join(path, method, filename))
  graph_df = pd.DataFrame({
    'from_index' : graph['edge_index'][0],
    'from_token' : [graph['tokens'][i] for i in graph['edge_index'][0]],
    'to_index' : graph['edge_index'][1],
    'to_token' : [graph['tokens'][i] for i in graph['edge_index'][1]],
  })
  for dimension in range(graph['edge_attr'].size(1)):
    graph_df[f'edge_weight_{dimension}'] = graph['edge_attr'][:, dimension].tolist()
  
  # Ignore 0.0 edges
  graph_df = graph_df[graph_df[f'edge_weight_{edge}'] > 0.0]

  weight_columns = [f'edge_weight_{dimension}'for dimension in range(graph['edge_attr'].size(1))]
  incoming = graph_df[['to_index', 'to_token'] + weight_columns].rename(columns = {'to_index' : 'id', 'to_token' : 'token'}).groupby(['id', 'token'])[weight_columns].sum()
  outgoing = graph_df[['from_index', 'from_token'] + weight_columns].rename(columns = {'from_index' : 'id', 'from_token' : 'token'}).groupby(['id', 'token'])[weight_columns].sum()
  nodes = incoming.add(outgoing, fill_value = 0).reset_index()

  pattern_D = r'\[D\]'
  pattern_T = r'\[T-\d+\]'
  mask = (
    # Exclude edges connected to [D]
    ~graph_df['from_token'].str.contains(pattern_D) &
    ~graph_df['to_token'].str.contains(pattern_D) &
    # Exclude self-loops of [T-i]
    ~((graph_df['from_token'] == graph_df['to_token']) & graph_df['from_token'].str.contains(pattern_T)) &
    # Only include edges connected to [T-i] nodes
    (graph_df['from_token'].str.contains(pattern_T) | graph_df['to_token'].str.contains(pattern_T))
  )
  edge_colors = pd.Series('#cccccc', index=graph_df.index)  # default grey for excluded edges
  edge_colors[mask] = create_color_gradient(graph_df.loc[mask, f'edge_weight_{edge}'], grey_to_black_cmap) # ALTER edge_weight_0
  node_colors = create_color_gradient(nodes[f'edge_weight_{edge}'], light_to_dark_yellow_cmap)

  graph = graphviz.Digraph(f'{filename.replace(".pt", "")}-{edge}', engine = 'dot') # engine = 'circo', 'twopi', 'dot
  graph.graph_attr['dpi'] = '300'

  with graph.subgraph() as highest_rank:
    highest_rank.attr(rank='same')
    for i, node in nodes[nodes['token'].str.contains(r'\[D\]')].iterrows():
      highest_rank.node(str(node['id']), label=node['token'], color='#363636', style='filled', fillcolor='#ffd53d', shape='oval')

  with graph.subgraph() as second_rank:
    second_rank.attr(rank='same')
    for i, node in nodes[nodes['token'].str.contains(r'\[T-\d+\]')].iterrows():
      second_rank.node(str(node['id']), label=node['token'], color='#363636', style='filled', fillcolor='#fffa7d', shape='oval')

  min_size = 0.5
  max_size = 2.0
  min_weight = nodes[f'edge_weight_{edge}'].min()
  max_weight = nodes[f'edge_weight_{edge}'].max()
  with graph.subgraph() as lowest_rank:
    lowest_rank.attr(rank='same')
    for i, node in nodes[~nodes['token'].str.contains(r'\[D\]|\[T-\d+\]')].iterrows():
      normalized = (node[f'edge_weight_{edge}'] - min_weight) / (max_weight - min_weight) if max_weight != min_weight else 0.5
      size = min_size + normalized * (max_size - min_size)
      lowest_rank.node(str(node['id']), label = node['token'], color = '#363636', style = 'filled', fillcolor = node_colors[i], shape = 'oval', width = str(2.5 * size), height = str(size), fixedsize = 'true')

  for i, row in graph_df.iterrows():
    graph.edge(str(row['from_index']), str(row['to_index']), arrowsize = '0.5', color = edge_colors[i])
  os.makedirs(os.path.join(STORAGE_PATH, method), exist_ok = True)
  graph.render(directory = os.path.join(STORAGE_PATH, method, f'{filename.replace(".pt", "")}-{edge}'), format = 'png', view = True)

In [41]:
visualize_graph(path = './graphs/', method = 'chefer_importance', filename = 'SST-2-test-20.pt', edge = 0)

In [42]:
visualize_graph(path = './graphs/', method = 'chefer_importance', filename = 'SST-2-test-20.pt', edge = 1)